In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from lesson_1.ingest import load_faq_data
import requests
from elasticsearch import Elasticsearch

In [3]:
documents = load_faq_data()
len(documents)

1349

In [4]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 84 documents


In [5]:
es = Elasticsearch("http://localhost:9200")

In [6]:
def fit(docs_llm, index_name="faq-llm-zoomcamp"):
    
    if not es.indices.exists(index=index_name):
        # index doesn't exist yet — create and index all
        es.options(ignore_status=400).indices.create(
            index=index_name,
            body={
                "mappings": {
                    "properties": {
                        "question": {"type": "text"},
                        "section":  {"type": "text"},
                        "answer":   {"type": "text"},
                        "course":   {"type": "keyword"}
                    }
                }
            }
        )
        for doc in docs_llm:
            es.index(index=index_name, document=doc)
        print(f"Created index and indexed {len(docs_llm)} docs")

    elif es.count(index=index_name)["count"] == len(docs_llm):
        # same number of docs — nothing changed, skip
        print(f"Index already up to date with {len(docs_llm)} docs, skipping")

    else:
        # count mismatch — wipe and re-index fresh
        es.options(ignore_status=[400, 404]).indices.delete(index=index_name)
        es.options(ignore_status=400).indices.create(
            index=index_name,
            body={
                "mappings": {
                    "properties": {
                        "question": {"type": "text"},
                        "section":  {"type": "text"},
                        "answer":   {"type": "text"},
                        "course":   {"type": "keyword"}
                    }
                }
            }
        )
        for doc in docs_llm:
            es.index(index=index_name, document=doc)
        print(f"Re-indexed {len(docs_llm)} docs")

In [7]:
# write the data into Elasticsearch, which then gets persisted in the Docker volume
fit(docs_llm, index_name="faq-llm-zoomcamp")

Created index and indexed 84 docs
